# Holdout test

The holdout answers one question: **how wrong is the estimate for a company the method was never adjusted on?** The 160 holdout firms are scored here, once, with everything frozen. `comps_and_valuation.ipynb` is the development notebook (training firms only); this one reads its saved split and applies the same rules to new targets.

**Two modes, set in section 1:**

- **`MODE = "training"` (default).** A reference run: the same rules on the training targets, with the same code path the holdout will use. It never reads a holdout multiple. Use it to check the notebook, and to get the training numbers the holdout is compared with.
- **`MODE = "holdout"`.** Needs `holdout/freeze.json` (written by this notebook, section 1) and refuses to score twice under the same freeze. A holdout target's comps come from the training firms only (D4); its multiple is read only in section 5.

**What freezing means.** Every setting that was chosen by looking at training results is written to `freeze.json` before any holdout multiple is read: the split run, facets version, embedding model, k and z levels, the judge model and prompt version, and the one rule reported as the headline (`PRIMARY_RULE`). If any of them differs from the file at scoring time, the notebook stops. Changing anything after seeing holdout results means a new freeze, and the earlier holdout result no longer counts as untouched.

**Why an interval.** On 160 targets, a gap of one or two points between rules is inside the noise (the training results already show it: changing the split alone moved the same rule by 2-4 points). Section 5 resamples the targets and reports whether a rule's gain over no matching has an interval that excludes 0.

**Not in this notebook:** the direct-LLM routes (`direct_llm.py`, `llm_then_cosine.py`) are not scored on the holdout. `direct_llm.py` takes the training firms as its targets, so it needs a target-set option first. The golden set and its metrics are in `golden_set.ipynb`.

## 0. Inputs

In [1]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# repo root = the folder holding data_manifest.json (works from the repo root, v0/ or v0/notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_manifest.json").exists())
sys.path.insert(0, str(ROOT / "v1" / "scripts"))
import facets as fc                     # also puts website_scrape/ and src/ on the path
import candidates as cd
import cosine_then_llm as jc
import valuation as vl
from fill_facets import DEFAULT_OUT as FACETS_OUT, output_paths
from project_data import OUTPUTS_LOCAL

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

RUN = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
COMPS_OUT = OUTPUTS_LOCAL / "v1"
WRITE_OUT = OUTPUTS_LOCAL / "v1"     # the development notebook's outputs; read here
HOLD_OUT = WRITE_OUT / "holdout"                      # this notebook writes here

UNIVERSE_RUN = sorted(p for p in (COMPS_OUT / "universe").iterdir() if (p / "universe.csv").exists())[-1]
universe = pd.read_csv(UNIVERSE_RUN / "universe.csv", dtype=str)
facets = pd.read_csv(output_paths(FACETS_OUT).tables / "facets.csv", dtype=str).fillna("")
facets = facets[(facets["prompt_version"] == fc.PROMPT_VERSION) & (facets["llm_status"] == "ok")]
financials = pd.read_csv(UNIVERSE_RUN / "financials.csv", dtype={"IQ_COMPANY_ID": str}).set_index("IQ_COMPANY_ID")
financials = financials.loc[universe["IQ_COMPANY_ID"]]
info = universe.set_index("IQ_COMPANY_ID")

# the split the development notebook saved (latest run; the freeze pins it)
SPLIT_RUN = sorted(p for p in (WRITE_OUT / "split").iterdir() if (p / "split.csv").exists())[-1]
split = pd.read_csv(SPLIT_RUN / "split.csv", dtype=str)
split_meta = json.loads((SPLIT_RUN / "run.json").read_text(encoding="utf-8"))
assert split_meta["universe_run"] == UNIVERSE_RUN.name and split_meta["facets_version"] == fc.PROMPT_VERSION, \
    "the saved split was made on a different universe or facets version"
assert set(split["IQ_COMPANY_ID"]) == set(universe["IQ_COMPANY_ID"])
training = split.loc[split["sample"] == "training", "IQ_COMPANY_ID"].tolist()
holdout = split.loc[split["sample"] == "holdout", "IQ_COMPANY_ID"].tolist()
print(f"split run {SPLIT_RUN.name} (seed {split_meta['seed']}, {split_meta['stratified_on']}): "
      f"{len(training)} training, {len(holdout)} holdout; universe run {UNIVERSE_RUN.name}; facets {fc.PROMPT_VERSION}")

split run 20261027T204930Z (seed 2026, SIC2 (groups under 6 firms merged into other)): 373 training, 160 holdout; universe run 20261025T060358Z; facets 2026-10-25k


## 1. Mode and freeze

`PRIMARY_RULE` is the one comp rule reported as the headline result. It has to be chosen on training results and written down **before** the holdout is scored; every other rule is still reported, but none of them can be promoted after the fact. Choose from `RULES` below. `WRITE_FREEZE = True` (with `PRIMARY_RULE` set) writes `holdout/freeze.json`; an earlier freeze file is moved to `holdout/history/`.

In [2]:
MODE = "training"        # "training" (reference run) or "holdout" (needs freeze.json; scores once)
PRIMARY_RULE = None      # the headline rule, chosen on training results with the team; one of RULES
WRITE_FREEZE = False     # True: write holdout/freeze.json from the settings below, then set back to False

# rule names; section 4 builds its results with the same names
NO_MATCH = "no matching (median of the training firms)"
CUTS = {"top 10": "in_top_k", **{f"z ≥ {lvl:g}": f"z_ge_{lvl:g}" for lvl in cd.Z_LEVELS}}
LLM_RULE = "LLM good comps (from z ≥ 2), similarity-weighted mean"
SIC_LEVELS = {"SIC4": "target_sic4_code", "SIC3": "target_sic3_code", "SIC2": "target_sic2_code"}
RULES = [NO_MATCH,
         *[f"cosine {cut}, {how}" for cut in CUTS for how in ("similarity-weighted mean", "median")],
         LLM_RULE, "LLM good comps (from z ≥ 2), median",
         *[f"same {level} (no embeddings, no LLM), mean" for level in SIC_LEVELS]]

SETTINGS = {"split_run": SPLIT_RUN.name, "universe_run": UNIVERSE_RUN.name, "facets_version": fc.PROMPT_VERSION,
            "embed_model": cd.EMBED_MODEL, "similarity_facets": cd.SIMILARITY_FACETS, "weights": "equal",
            "top_k": cd.TOP_K, "z_levels": list(cd.Z_LEVELS), "pool": "training",
            "judge_model": jc.DEFAULT_MODEL["claude"], "judge_prompt_version": jc.SAVED_PROMPT_VERSION, "judge_z": jc.DEFAULT_Z,
            "rules": RULES, "primary_rule": PRIMARY_RULE}


def git(*args: str) -> str:
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True).stdout.strip()


FREEZE_PATH = HOLD_OUT / "freeze.json"
if WRITE_FREEZE:
    if PRIMARY_RULE not in RULES:
        raise SystemExit(f"Set PRIMARY_RULE to one of RULES before freezing (now {PRIMARY_RULE!r}).")
    HOLD_OUT.mkdir(parents=True, exist_ok=True)
    if FREEZE_PATH.exists():
        (HOLD_OUT / "history").mkdir(exist_ok=True)
        FREEZE_PATH.rename(HOLD_OUT / "history" / f"freeze_{RUN}.json")
    FREEZE_PATH.write_text(json.dumps({"frozen_at": RUN, "settings": SETTINGS, "git_commit": git("rev-parse", "HEAD"),
                                       "git_uncommitted_in_v0": bool(git("status", "--porcelain", "--", "v1"))},
                                      indent=1, ensure_ascii=False), encoding="utf-8")
    print("wrote", FREEZE_PATH.relative_to(ROOT))
frozen = json.loads(FREEZE_PATH.read_text(encoding="utf-8")) if FREEZE_PATH.exists() else None

if MODE == "holdout":
    if frozen is None:
        raise SystemExit("No freeze yet: set PRIMARY_RULE, run once with WRITE_FREEZE = True, then set it back to False.")
    changed = {k: (frozen["settings"].get(k), v) for k, v in SETTINGS.items() if frozen["settings"].get(k) != v}
    if changed:
        raise SystemExit(f"Settings differ from the freeze {frozen['frozen_at']} (frozen, now): {changed}")
    scored = [p.parent.name for p in (HOLD_OUT / "scores").glob("*/run.json")
              if json.loads(p.read_text(encoding="utf-8")).get("freeze") == frozen["frozen_at"]]
    if scored:
        raise SystemExit(f"The holdout was already scored under freeze {frozen['frozen_at']} (run {scored[0]}). "
                         "A new score needs a new freeze, and the earlier result stays on record as seen.")
elif MODE != "training":
    raise ValueError(f"MODE must be 'training' or 'holdout', not {MODE!r}")

targets = training if MODE == "training" else holdout
print(f"MODE {MODE}: {len(targets)} targets, comps from the {len(training)} training firms"
      + (f"; freeze {frozen['frozen_at']}, primary rule: {frozen['settings']['primary_rule']}" if MODE == "holdout" else
         f"; freeze on record: {frozen['frozen_at'] if frozen else 'none'}"))

MODE training: 373 targets, comps from the 373 training firms; freeze on record: none


## 2. Candidates (no multiples, no API)

The same step 4 as the development notebook: per-facet embeddings (cached), equal weights, company size left out. The ranking is built for this mode's targets against the **training** pool. A holdout target's z is taken over the training firms, as a training target's is over the other training firms. Only facets are read here, never a multiple.

In [3]:
texts = cd.facet_texts(facets).reindex(universe["IQ_COMPANY_ID"])
ids = list(texts.index)
embeddings, embed_cache = cd.embed_facets(texts, COMPS_OUT / "embeddings")
S, per_facet = cd.similarity(embeddings)
ranking = cd.rank_candidates(S, ids, targets=targets, pool=training)
cands = cd.add_facet_similarities(cd.candidates(ranking), per_facet, ids)
for col, src in [("target_ticker", "target"), ("comp_ticker", "comp")]:
    cands.insert(cands.columns.get_loc(src) + 1, col, cands[src].map(info["target_ciq_ticker"]))
cands.insert(cands.columns.get_loc("comp_ticker") + 1, "comp_company", cands["comp"].map(info["target_company_name"]))

z_cols = [f"z_ge_{lvl:g}" for lvl in cd.Z_LEVELS]
per_target_n = ranking.groupby("target")[z_cols].sum()
print(f"{ranking['target'].nunique()} targets x {len(training)} pool firms; embeddings {embed_cache.name}")
for col in z_cols:
    n = per_target_n[col]
    print(f"{col}: median {n.median():.0f} comps per target, range {n.min()}-{n.max()}; {(n < 3).sum()} targets with fewer than 3")
print(f"candidates (top {cd.TOP_K} or z >= {min(cd.Z_LEVELS):g}): {len(cands)} pairs, "
      f"{cands.groupby('target').size().median():.0f} per target (median)")

if MODE == "holdout":      # the run folder cosine_then_llm.py reads; facets only, no multiples
    CAND_OUT = HOLD_OUT / "candidates" / frozen["frozen_at"]
    CAND_OUT.mkdir(parents=True, exist_ok=True)
    cands.to_csv(CAND_OUT / "candidates.csv", index=False)
    ranking.to_csv(CAND_OUT / "ranking.csv.gz", index=False)
    (CAND_OUT / "run.json").write_text(json.dumps({
        "run": frozen["frozen_at"], "split_run": SPLIT_RUN.name, "universe_run": UNIVERSE_RUN.name,
        "facets_version": fc.PROMPT_VERSION, "embed_model": cd.EMBED_MODEL, "embedding_cache": embed_cache.name,
        "targets": "holdout", "pool": "training", "top_k": cd.TOP_K, "z_levels": list(cd.Z_LEVELS), "pairs": len(cands),
    }, indent=1), encoding="utf-8")
    print("wrote", CAND_OUT.relative_to(ROOT))

373 targets x 373 pool firms; embeddings facets_all-mpnet-base-v2_6f4ce4373bfbc08a.npz
z_ge_2: median 13 comps per target, range 5-25; 0 targets with fewer than 3
z_ge_2.5: median 6 comps per target, range 0-15; 41 targets with fewer than 3
candidates (top 10 or z >= 2): 5110 pairs, 13 per target (median)


## 3. The LLM judge (recipe step 6)

The same step as the development notebook: `cosine_then_llm.py` marks each z ≥ 2 candidate good or bad. In training mode this reads the judgments the development notebook already has. In holdout mode the judge is run **once, on the frozen prompt and model**, against the candidates written in section 2; this cell stops and prints the command if any holdout target is missing a judgment. That is the only API call the holdout needs, and it is made after the freeze.

In [4]:
JUDGE_DIR = WRITE_OUT / "judge" if MODE == "training" else HOLD_OUT / "judge"
JUDGE_TABLE = jc.output_paths(JUDGE_DIR).tables / "judgments.csv"
judgments = (pd.read_csv(JUDGE_TABLE, dtype={"target": str, "comp": str}) if JUDGE_TABLE.exists()
             else pd.DataFrame(columns=["target", "comp", "score", "reason", "model", "prompt_version", "candidates_run"]))
judgments = judgments[(judgments["prompt_version"] == jc.SAVED_PROMPT_VERSION) & (judgments["model"] == SETTINGS["judge_model"])]
if MODE == "holdout":
    judgments = judgments[judgments["candidates_run"] == CAND_OUT.name]
z2 = ranking[ranking[f"z_ge_{jc.DEFAULT_Z:g}"]][["target", "comp", "rank", "similarity", "z"]]
judged = z2.merge(judgments[["target", "comp", "score", "reason"]], on=["target", "comp"], how="left")
complete = judged.groupby("target")["score"].apply(lambda s: s.notna().all())
missing = sorted(set(targets) - set(complete[complete].index))
if MODE == "holdout" and missing:
    raise SystemExit(f"{len(missing)} of {len(targets)} holdout targets have no judgment yet. Run the frozen judge once:\n"
                     f"  python v1/scripts/cosine_then_llm.py --candidates {CAND_OUT.relative_to(ROOT)} "
                     f"--out {JUDGE_DIR.relative_to(ROOT)} --prompt-version {jc.SAVED_PROMPT_VERSION} --workers 4\nthen run this notebook again.")
judged = judged[judged["target"].isin(complete[complete].index)]
print(f"{MODE}: {complete.sum()} of {len(targets)} targets judged (prompt {jc.SAVED_PROMPT_VERSION}), {len(judged)} pairs, "
      f"{int(judged['score'].sum())} good ({judged['score'].mean():.0%}); "
      f"median {judged.groupby('target')['score'].sum().median():.0f} good comps per target")

training: 373 of 373 targets judged (prompt 2026-10-25a), 4996 pairs, 592 good (12%); median 1 good comps per target


## 4. Estimates

Each rule turns a target's comps into an estimate: a similarity-weighted mean (or median) of the comps' EV/EBITDA, as in the development notebook. Comps' multiples come from the **training** firms only (`ev_pool`), so a holdout multiple cannot reach an estimate. The baseline, no matching, is the median multiple of the training pool. The target's own multiple (the answer key) is read in section 5.

In [5]:
ev_pool = financials["target_ev_to_ebitda"].reindex(training)     # comps' multiples: training firms only
results = {NO_MATCH: vl.no_matching(targets, training, ev_pool)}
for cut, col in CUTS.items():
    pairs = ranking[ranking[col]]
    results[f"cosine {cut}, similarity-weighted mean"] = vl.estimate(pairs, ev_pool, weight="similarity")
    results[f"cosine {cut}, median"] = vl.estimate(pairs, ev_pool, how="median")
good = judged[judged["score"] == 1]
if len(good):
    results[LLM_RULE] = vl.estimate(good, ev_pool, weight="similarity")
    results["LLM good comps (from z ≥ 2), median"] = vl.estimate(good, ev_pool, how="median")
tg, tr = info.loc[targets].reset_index(), info.loc[training].reset_index()
for level, col in SIC_LEVELS.items():                              # same-SIC training peers, no model at all
    m = tg[["IQ_COMPANY_ID", col]].merge(tr[["IQ_COMPANY_ID", col]], on=col, suffixes=("_t", "_c"))
    m = m[m["IQ_COMPANY_ID_t"] != m["IQ_COMPANY_ID_c"]].rename(columns={"IQ_COMPANY_ID_t": "target", "IQ_COMPANY_ID_c": "comp"})
    results[f"same {level} (no embeddings, no LLM), mean"] = vl.estimate(m, ev_pool)
assert set(results) <= set(RULES), set(results) - set(RULES)
if MODE == "holdout":
    assert set(results) == set(RULES), f"rules without an estimate: {set(RULES) - set(results)}"
results = {name: results[name] for name in RULES if name in results}
print(f"{len(results)} rules; targets with an estimate: " + ", ".join(f"{len(r)}" for r in results.values()))

12 rules; targets with an estimate: 373, 373, 373, 373, 373, 368, 368, 264, 264, 287, 332, 367


## 5. Scores against the actual multiples

This is where the target's own EV/EBITDA is first read (`actual`). In holdout mode, that is the first time any holdout multiple is used anywhere in the method.

**Measures:** median absolute % error, % of targets within 25%, Spearman (estimate, actual), and % of targets closer than with no matching, as in the development notebook. The second table gives a 95% bootstrap interval for each rule's error and for its **gain on no matching** (points of median absolute % error; an interval that includes 0 means the rule is not shown to beat no matching).

In [6]:
actual = financials["target_ev_to_ebitda"].reindex(targets)
evaluation = vl.summary(results, actual, baseline=NO_MATCH)
print(f"{MODE}: {len(targets)} targets, comps from the {len(training)} training firms")
evaluation.round(2)

training: 373 targets, comps from the 373 training firms


,targets,comps per target (median),median abs % error,% within 25%,"Spearman(estimate, actual)",% closer than no matching
comp rule,,,,,,
no matching (median of the training firms),373,372.0,31.92,38.87,NaN,NaN
"cosine top 10, similarity-weighted mean",373,10.0,28.82,42.90,0.35,50.94
"cosine top 10, median",373,10.0,31.01,41.29,0.31,52.28
"cosine z ≥ 2, similarity-weighted mean",373,13.0,29.58,42.63,0.31,49.33
"cosine z ≥ 2, median",373,13.0,31.10,38.34,0.25,51.74
"cosine z ≥ 2.5, similarity-weighted mean",368,6.0,30.23,42.66,0.33,51.36
"cosine z ≥ 2.5, median",368,6.0,32.10,41.30,0.30,51.63
"LLM good comps (from z ≥ 2), similarity-weighted mean",264,2.0,30.50,40.91,0.44,50.00
"LLM good comps (from z ≥ 2), median",264,2.0,30.50,40.53,0.42,50.38


In [7]:
intervals = vl.bootstrap(results, actual, baseline=NO_MATCH)
intervals.round(1)

,targets,median abs % error,error 2.5%,error 97.5%,gain on baseline,gain 2.5%,gain 97.5%
comp rule,,,,,,,
no matching (median of the training firms),373,31.9,29.4,34.4,NaN,NaN,NaN
"cosine top 10, similarity-weighted mean",373,28.8,26.3,34.0,3.1,-1.5,5.5
"cosine top 10, median",373,31.0,27.2,33.2,0.9,-0.9,4.2
"cosine z ≥ 2, similarity-weighted mean",373,29.6,26.1,33.7,2.3,-0.9,5.3
"cosine z ≥ 2, median",373,31.1,27.8,35.3,0.8,-2.5,3.5
"cosine z ≥ 2.5, similarity-weighted mean",368,30.2,26.6,34.7,1.7,-2.0,5.5
"cosine z ≥ 2.5, median",368,32.1,27.9,35.7,-0.1,-2.9,4.4
"LLM good comps (from z ≥ 2), similarity-weighted mean",264,30.5,27.0,33.9,1.1,-3.0,6.1
"LLM good comps (from z ≥ 2), median",264,30.5,26.5,33.3,1.1,-3.1,6.3


The same rules on the targets that have an LLM-judged estimate, so every row is scored on the same companies.

In [8]:
covered = results[LLM_RULE].index if LLM_RULE in results else pd.Index([])
evaluation_covered = (vl.summary({k: v.loc[v.index.intersection(covered)] for k, v in results.items()},
                                 actual[covered], baseline=NO_MATCH) if len(covered) else pd.DataFrame())
print(f"{len(covered)} of {len(targets)} targets have an LLM-judged estimate")
evaluation_covered.round(2)

264 of 373 targets have an LLM-judged estimate


,targets,comps per target (median),median abs % error,% within 25%,"Spearman(estimate, actual)",% closer than no matching
comp rule,,,,,,
no matching (median of the training firms),264,372.0,31.64,40.91,NaN,NaN
"cosine top 10, similarity-weighted mean",264,10.0,28.31,44.32,0.38,52.27
"cosine top 10, median",264,10.0,30.44,40.15,0.33,53.41
"cosine z ≥ 2, similarity-weighted mean",264,13.0,28.71,45.08,0.33,48.86
"cosine z ≥ 2, median",264,13.0,30.97,38.64,0.27,52.27
"cosine z ≥ 2.5, similarity-weighted mean",264,6.0,29.47,43.56,0.37,53.79
"cosine z ≥ 2.5, median",264,6.0,32.11,40.15,0.32,53.79
"LLM good comps (from z ≥ 2), similarity-weighted mean",264,2.0,30.50,40.91,0.44,50.00
"LLM good comps (from z ≥ 2), median",264,2.0,30.50,40.53,0.42,50.38


### Training against holdout

Holdout mode only: the training reference run saved by this notebook (`MODE = "training"`, `holdout/reference_training/`), next to the holdout scores, for the rules both have. A holdout error well above the training error means the settings were fit to the training firms.

In [9]:
if MODE == "holdout":
    refs = sorted((HOLD_OUT / "reference_training").glob("*/summary.csv"))
    if not refs:
        print("No training reference run saved yet: run this notebook with MODE = 'training' first.")
    else:
        ref = pd.read_csv(refs[-1], index_col=0)
        cols = ["targets", "median abs % error", "% within 25%", "Spearman(estimate, actual)", "% closer than no matching"]
        side = ref[cols].add_prefix("training: ").join(evaluation[cols].add_prefix("holdout: "), how="inner")
        print(f"training reference run {refs[-1].parent.name}; primary rule: {frozen['settings']['primary_rule']}")
        display(side.round(2))
else:
    print("Training mode: nothing to compare yet.")

Training mode: nothing to compare yet.


## 6. Save

In [10]:
SCORE_OUT = (HOLD_OUT / "reference_training" if MODE == "training" else HOLD_OUT / "scores") / RUN
SCORE_OUT.mkdir(parents=True, exist_ok=True)
per_target = pd.concat({name: vl.errors(r, actual) for name, r in results.items()},
                       names=["comp rule", "IQ_COMPANY_ID"]).reset_index()
per_target.insert(2, "ticker", per_target["IQ_COMPANY_ID"].map(info["target_ciq_ticker"]))
evaluation.to_csv(SCORE_OUT / "summary.csv")
evaluation_covered.to_csv(SCORE_OUT / "summary_llm_covered.csv")
intervals.to_csv(SCORE_OUT / "intervals.csv")
per_target.to_csv(SCORE_OUT / "per_target.csv", index=False)
(SCORE_OUT / "run.json").write_text(json.dumps({
    "run": RUN, "mode": MODE, "freeze": frozen["frozen_at"] if MODE == "holdout" else None,
    "settings": SETTINGS, "targets": len(targets), "pool": f"training ({len(training)} firms)",
    "baseline": NO_MATCH, "rules": list(results), "llm_targets_judged": int(complete.sum()),
    "bootstrap": {"n_boot": 2000, "seed": 2026}, "git_commit": git("rev-parse", "HEAD"),
}, indent=1, ensure_ascii=False), encoding="utf-8")
print("wrote", SCORE_OUT.relative_to(ROOT))

wrote outputs-local\comps_pilot_targets\holdout\reference_training\20261027T215012Z
